# Phân tích yếu tố nguy cơ bệnh tim mạch
## 1. Introduction

### 1.1 Bối cảnh

Bệnh tim mạch là một trong những nguyên nhân gây tử vong hàng đầu trên thế giới.
Việc dự đoán sớm nguy cơ tim mạch từ dữ liệu lâm sàng cơ bản (tuổi, huyết áp,
BMI, lối sống, xét nghiệm cơ bản) có ý nghĩa lớn trong phòng ngừa và can thiệp sớm.

Dữ liệu bài toán có dạng **tabular clinical data**, trong đó:
- Quan hệ tuyến tính (linear trend) tồn tại rõ ràng với các biến như tuổi, huyết áp.
- Đồng thời, cũng có các **tương tác phi tuyến** (non-linear interactions) giữa
  tuổi – BMI – huyết áp.

Do đó, việc kết hợp mô hình tuyến tính (Logistic Regression) và mô hình boosting
(CatBoost, XGBoost) là một lựa chọn phù hợp.

### 1.2 Mục tiêu nghiên cứu

**Primary goal:**  
Đánh giá liệu **Soft Voting Ensemble** giữa Logistic Regression, CatBoost và
XGBoost có mang lại **cải thiện ổn định** so với từng mô hình đơn lẻ hay không,
thông qua ROC-AUC cross-validation (mean ± std).

**Stretch goal:**  
Tiệm cận hoặc đạt **ROC-AUC ≥ 0.80** thông qua feature engineering
(pulse pressure, interaction features) và ensemble learning.

### 1.3 Câu hỏi nghiên cứu

- Việc kết hợp mô hình tuyến tính và boosting bằng soft voting có cải thiện AUC
  một cách nhất quán không?
- Những interaction feature nào đóng góp đáng kể vào hiệu suất mô hình?
- Sau khi mô hình dự đoán, các ngưỡng tuổi, huyết áp và BMI nào làm nguy cơ tim mạch tăng mạnh
  (post-hoc risk stratification, không gây data leakage)?

### 1.4 Đóng góp chính của bài

- So sánh có kiểm soát giữa Logistic Regression, CatBoost, XGBoost và Soft Voting Ensemble.
- Xử lý đa cộng tuyến đúng vị trí (tập trung ở Logistic Regression).
- Phân tích hậu kiểm (post-hoc) ngưỡng rủi ro dựa trên **out-of-fold predictions**,
  đảm bảo không rò rỉ dữ liệu.
- Báo cáo đầy đủ cả **discrimination (AUC)** và **calibration (Brier score, calibration curve)**.


## 1. Import thư viện

In [ ]:
# Reproducibility
RANDOM_STATE = 42

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# EDA tự động
!pip install ydata-profiling > /dev/null
from ydata_profiling import ProfileReport

# Sklearn core
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

# Ensemble
from sklearn.ensemble import VotingClassifier

# Set global random seed
np.random.seed(RANDOM_STATE)

print("Environment setup completed. Random state =", RANDOM_STATE)



## 2. Dataset & Problem Setup

### 2.1 Dataset description

Bộ dữ liệu được sử dụng là **Cardiovascular Disease Dataset**, bao gồm các đặc trưng
lâm sàng và hành vi cơ bản của bệnh nhân.

- Mỗi dòng tương ứng với một cá nhân
- Target: `cardio`
  - 0: không mắc bệnh tim mạch
  - 1: có bệnh tim mạch

Dữ liệu đã được kiểm tra tổng quan bằng EDA tự động (ProfileReport), do đó phần này
chỉ tập trung vào setup cho mô hình hóa.

### 2.2 Data dictionary (rút gọn)

| Feature        | Mô tả |
|----------------|------|
| age            | Tuổi (đơn vị: ngày, sẽ chuyển sang năm) |
| height         | Chiều cao (cm) |
| weight         | Cân nặng (kg) |
| ap_hi          | Huyết áp tâm thu (mmHg) |
| ap_lo          | Huyết áp tâm trương (mmHg) |
| cholesterol    | Cholesterol (ordinal) |
| gluc           | Glucose (ordinal) |
| smoke          | Hút thuốc (0/1) |
| alco           | Uống rượu (0/1) |
| active         | Hoạt động thể chất (0/1) |
| cardio         | Nhãn bệnh tim mạch (target) |

### 2.3 Task definition

Bài toán **binary classification**:
> Dự đoán xác suất một cá nhân mắc bệnh tim mạch (`cardio = 1`).

### 2.4 Evaluation metrics

- **Primary metric:** ROC-AUC
- **Secondary metrics:**
  - Brier score (đánh giá calibration)
  - Calibration curve
- **Stability / Uncertainty:**
  - Stratified K-Fold cross-validation
  - Báo cáo AUC dưới dạng mean ± std

### 2.5 Split strategy (fair comparison)

- Chia dữ liệu thành **train / test** theo tỉ lệ 80/20
- **Stratified split** theo biến `cardio`
- Cố định `random_state` để tái lập kết quả
- Test set **chỉ dùng cho báo cáo kết quả cuối**
- Toàn bộ feature engineering, CV, threshold selection, post-hoc analysis
  **chỉ thực hiện trên train set hoặc OOF predictions**


In [ ]:
# Load dataset
DATA_PATH = "cardio_train.csv"
df = pd.read_csv(DATA_PATH, sep=";")

print("Dataset shape:", df.shape)
print("Target distribution:")
print(df["cardio"].value_counts(normalize=True))
df.head()

In [ ]:
# Separate features and target
X = df.drop(columns=["cardio"])
y = df["cardio"]

# Train / test split (stratified, fixed seed)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

## 3. Appendix A – Exploratory Data Analysis (EDA)

EDA được thực hiện tự động bằng thư viện `ydata-profiling`
nhằm kiểm tra:
- Phân phối dữ liệu
- Missing values
- Outliers
- Tương quan cơ bản giữa các biến

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df,
    title="Cardiovascular Disease Dataset – EDA",
    explorative=True
)

# Xuất ra file HTML
profile.to_file("cardiovascular_eda.html")

## 3. Data Preprocessing

### 3.1 Mục tiêu preprocessing

Mục tiêu của bước tiền xử lý dữ liệu là:
- Chuẩn hóa đơn vị và tạo các biến cơ bản phục vụ mô hình hóa
- Loại bỏ các giá trị **vô lý về mặt sinh lý học**
- Kiểm soát ảnh hưởng của các giá trị cực đoan (outliers) **mà không loại bỏ quan sát**
- Đảm bảo toàn bộ quy trình **không gây rò rỉ dữ liệu (data leakage)**

Toàn bộ các phép biến đổi trong bước này:
- Được **fit trên tập train**
- Và chỉ **apply cho tập test**

### 3.2 Làm sạch dữ liệu dựa trên logic sinh lý học

Một số giá trị trong dữ liệu có thể không hợp lý về mặt y khoa
(do lỗi đo lường hoặc nhập liệu). Các quan sát này được loại bỏ
dựa trên các ràng buộc sinh lý học sau:

- Huyết áp tâm thu (`ap_hi`) nằm trong khoảng [90, 240] mmHg
- Huyết áp tâm trương (`ap_lo`) nằm trong khoảng [60, 160] mmHg
- Luôn đảm bảo `ap_hi > ap_lo`

Bước này nhằm đảm bảo chất lượng dữ liệu đầu vào,  
không được xem là loại bỏ outlier theo nghĩa thống kê.

### 3.3 Xử lý outlier theo hướng bền vững (mild winsorization)

Thay vì loại bỏ outlier dựa trên tiêu chí thống kê thuần túy
(ví dụ IQR hoặc z-score), nghiên cứu này áp dụng **winsorization nhẹ**
để **kiểm soát ảnh hưởng của các giá trị cực đoan** mà vẫn giữ lại toàn bộ quan sát.

Cụ thể:
- Áp dụng winsorization ở mức **0.5% – 99.5%**
- Chỉ áp dụng cho các biến continuous:
  - Tuổi (`age_years`)
  - Chỉ số khối cơ thể (`BMI`)
  - Huyết áp tâm thu (`ap_hi`)
  - Huyết áp tâm trương (`ap_lo`)
  - Áp lực mạch (`pulse_pressure`)
- Các ngưỡng percentile được **ước lượng trên tập train**
- Sau đó được áp dụng cho tập test bằng cùng ngưỡng

Cách tiếp cận này giúp:
- Giảm ảnh hưởng leverage của các giá trị cực đoan lên Logistic Regression
- Không làm mất các trường hợp rủi ro cao có ý nghĩa lâm sàng
- Giữ nguyên số lượng quan sát trong dữ liệu

### 3.4 Preprocessing theo từng mô hình

Do đặc tính khác nhau của các mô hình, bước tiền xử lý được thiết kế phù hợp:

- **Logistic Regression**:
  - Chuẩn hóa dữ liệu bằng `StandardScaler`
  - Sử dụng regularization L2 để tăng độ ổn định
- **CatBoost và XGBoost**:
  - Không cần chuẩn hóa dữ liệu
  - Tự xử lý phân tách ngưỡng cho các biến continuous

Việc tách biệt preprocessing theo mô hình giúp đảm bảo
so sánh công bằng và tránh xử lý dư thừa không cần thiết.


In [ ]:
def basic_preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    Basic preprocessing:
    - age: days -> years
    - compute BMI
    - remove physiologically implausible blood pressure values
    """
    df = df.copy()

    # Age: days -> years
    df["age_years"] = df["age"] / 365.25

    # BMI = weight (kg) / height (m)^2
    height_m = df["height"] / 100
    df["bmi"] = df["weight"] / (height_m ** 2)

    # Remove implausible blood pressure values
    bp_mask = (
        (df["ap_hi"] >= 90) & (df["ap_hi"] <= 240) &
        (df["ap_lo"] >= 60) & (df["ap_lo"] <= 160) &
        (df["ap_hi"] > df["ap_lo"])
    )

    df = df.loc[bp_mask]

    return df

In [ ]:
# Apply preprocessing
X_train_prep = basic_preprocess(X_train)
X_test_prep = basic_preprocess(X_test)

# Align targets using ORIGINAL index
y_train_prep = y_train.loc[X_train_prep.index]
y_test_prep = y_test.loc[X_test_prep.index]

# Reset index AFTER alignment
X_train_prep = X_train_prep.reset_index(drop=True)
y_train_prep = y_train_prep.reset_index(drop=True)

X_test_prep = X_test_prep.reset_index(drop=True)
y_test_prep = y_test_prep.reset_index(drop=True)

print("Train shapes:", X_train_prep.shape, y_train_prep.shape)
print("Test shapes :", X_test_prep.shape, y_test_prep.shape)
df.head()

In [ ]:
def fit_winsorizer(
    df: pd.DataFrame,
    cols: list,
    lower_q: float = 0.005,
    upper_q: float = 0.995
) -> dict:
    """
    Tính ngưỡng winsorization trên TRAIN SET
    """
    bounds = {}
    for col in cols:
        lower = df[col].quantile(lower_q)
        upper = df[col].quantile(upper_q)
        bounds[col] = (lower, upper)
    return bounds


def apply_winsorizer(
    df: pd.DataFrame,
    bounds: dict
) -> pd.DataFrame:
    """
    Áp dụng winsorization với ngưỡng đã fit
    """
    df = df.copy()
    for col, (lower, upper) in bounds.items():
        df[col] = df[col].clip(lower=lower, upper=upper)
    return df


In [ ]:
# Các biến continuous cần winsorize
WINSOR_COLS = [
    "age_years",
    "bmi",
    "ap_hi",
    "ap_lo",
]

# Fit winsorizer trên TRAIN
winsor_bounds = fit_winsorizer(
    X_train_prep,
    cols=WINSOR_COLS,
    lower_q=0.005,
    upper_q=0.995
)

# Apply cho cả train và test
X_train_clean = apply_winsorizer(X_train_prep, winsor_bounds)
X_test_clean = apply_winsorizer(X_test_prep, winsor_bounds)

print("Winsorization applied.")


In [ ]:
# Kiểm tra nhanh min/max sau winsorization
summary = pd.DataFrame({
    "train_min": X_train_clean[WINSOR_COLS].min(),
    "train_max": X_train_clean[WINSOR_COLS].max(),
})

summary


## 4. Feature Engineering

### 4.1 Mục tiêu

Mục tiêu của bước feature engineering là:
- Giữ các biến continuous quan trọng ở dạng liên tục (continuous modeling)
- Tạo thêm các biến có ý nghĩa sinh lý học
- Bổ sung interaction features để capture hiệu ứng phi tuyến
- Chuẩn bị bộ feature nhất quán cho tất cả mô hình

### 4.2 Continuous & derived features

Các biến continuous chính:
- age_years
- ap_hi, ap_lo
- bmi

Biến dẫn xuất:
- pulse_pressure = ap_hi − ap_lo

### 4.3 Centered interaction features

Trước khi tạo interaction:
- Các biến continuous được **centered** (trừ mean trên train set)

Interaction được tạo:
- age × BMI
- age × ap_hi
- BMI × ap_hi

Việc centering giúp:
- Giảm đa cộng tuyến
- Ổn định Logistic Regression
- Giữ ý nghĩa thống kê của hệ số


In [ ]:
def add_pulse_pressure(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tạo pulse pressure = ap_hi - ap_lo
    """
    df = df.copy()
    df["pulse_pressure"] = df["ap_hi"] - df["ap_lo"]
    return df


X_train_fe = add_pulse_pressure(X_train_clean)
X_test_fe = add_pulse_pressure(X_test_clean)

print("Pulse pressure added.")

In [ ]:
# Các biến continuous dùng cho interaction
CONT_COLS = [
    "age_years",
    "bmi",
    "ap_hi",
]

# Tính mean trên TRAIN
centering_values = {
    col: X_train_fe[col].mean()
    for col in CONT_COLS
}

def apply_centering(df: pd.DataFrame, centers: dict) -> pd.DataFrame:
    """
    Center các biến continuous bằng mean của TRAIN SET
    """
    df = df.copy()
    for col, mean_val in centers.items():
        df[f"{col}_c"] = df[col] - mean_val
    return df


X_train_fe = apply_centering(X_train_fe, centering_values)
X_test_fe = apply_centering(X_test_fe, centering_values)

print("Centering applied using TRAIN means.")


In [ ]:
def add_interactions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tạo interaction features từ các biến đã center
    """
    df = df.copy()

    df["age_bmi"] = df["age_years_c"] * df["bmi_c"]
    df["age_ap_hi"] = df["age_years_c"] * df["ap_hi_c"]
    df["bmi_ap_hi"] = df["bmi_c"] * df["ap_hi_c"]

    return df


X_train_fe = add_interactions(X_train_fe)
X_test_fe = add_interactions(X_test_fe)

print("Interaction features added.")

In [ ]:
BASE_FEATURES = [
    "age_years",
    "bmi",
    "ap_hi",
    "ap_lo",
    "pulse_pressure",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active",
]

INTERACTION_FEATURES = [
    "age_bmi",
    "age_ap_hi",
    "bmi_ap_hi",
]

FEATURES = BASE_FEATURES + INTERACTION_FEATURES

print("Total features used:", len(FEATURES))
FEATURES

## 5. Multicollinearity Handling

### 5.1 Vì sao đa cộng tuyến quan trọng?

Khi bổ sung interaction features, các biến:
- biến gốc
- biến đã center
- biến tương tác

có thể có tương quan cao với nhau.

Điều này:
- Ít ảnh hưởng đến tree-based models (CatBoost, XGBoost)
- Nhưng **ảnh hưởng trực tiếp đến Logistic Regression**:
  - Hệ số không ổn định
  - Odds ratio khó diễn giải

### 5.2 Chiến lược xử lý

- Centering trước khi tạo interaction (đã làm ở Bước 4)
- Logistic Regression sử dụng **L2 regularization**
- Chẩn đoán bằng:
  - Correlation matrix
  - Variance Inflation Factor (VIF)

Lưu ý:
- Đây là bước **diagnostic**, không phải feature selection bắt buộc
- Không loại feature trừ khi VIF quá cao và không có ý nghĩa lâm sàng


In [ ]:
# Feature set dùng để chẩn đoán đa cộng tuyến (giống LR input)
LR_FEATURES = FEATURES.copy()

X_lr = X_train_fe[LR_FEATURES].copy()

print("Number of LR features:", X_lr.shape[1])

In [ ]:
plt.figure(figsize=(5, 3))
corr = X_lr.corr()
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)
plt.title("Correlation Matrix (LR Feature Set)")
plt.show()

**Nhận xét (Correlation Matrix):**

- Các biến `ap_hi`, `ap_lo` và `pulse_pressure` có tương quan rất cao.
  Đây là hiện tượng **kỳ vọng trước**, vì `pulse_pressure` được định nghĩa trực tiếp
  từ hai biến huyết áp (`ap_hi − ap_lo`).
- Các biến liên tục khác như `age_years` và `bmi` có tương quan mức vừa phải
  với huyết áp, phù hợp với hiểu biết lâm sàng.
- Các biến hành vi (`smoke`, `alco`, `active`) gần như độc lập với nhóm huyết áp,
  không cho thấy dấu hiệu đa cộng tuyến nghiêm trọng.
- Các interaction features (`age_bmi`, `age_ap_hi`, `bmi_ap_hi`) không thể hiện
  tương quan tuyến tính cao với các biến gốc, cho thấy **centering trước khi tạo
  interaction đã phát huy hiệu quả**.


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tính Variance Inflation Factor (VIF)
    """
    vif_data = []
    X_values = df.values

    for i, col in enumerate(df.columns):
        vif = variance_inflation_factor(X_values, i)
        vif_data.append({
            "feature": col,
            "VIF": vif
        })

    return pd.DataFrame(vif_data).sort_values("VIF", ascending=False)


vif_table = compute_vif(X_lr)
vif_table

**Nhận xét (Variance Inflation Factor – VIF):**

- Các biến `ap_hi`, `ap_lo` và `pulse_pressure` có VIF vô hạn,
  do tồn tại **perfect linear dependency**:
  `pulse_pressure = ap_hi − ap_lo`.
  Đây là đa cộng tuyến **có chủ đích**, không phải lỗi mô hình hóa.
- Các biến trung tâm như `age_years` và `bmi` có VIF cao,
  điều này là chấp nhận được trong bối cảnh:
  - có interaction features
  - Logistic Regression sử dụng **L2 regularization**
- Các interaction features (`age_bmi`, `age_ap_hi`, `bmi_ap_hi`) có VIF gần 1,
  cho thấy centering đã **giảm đáng kể đa cộng tuyến** giữa interaction và biến gốc.

**Quyết định mô hình hóa:**  
Không loại bỏ feature nào ở bước này để:
- giữ đầy đủ ý nghĩa lâm sàng của các biến
- ưu tiên hiệu suất dự đoán
- kiểm soát đa cộng tuyến thông qua regularization thay vì loại biến.


## 6. Models

Trong bước này, ba mô hình cơ sở được xây dựng và đánh giá độc lập:

- Logistic Regression: baseline tuyến tính, dễ diễn giải
- CatBoost: boosting trên decision trees, mạnh với interaction phi tuyến
- XGBoost: boosting phổ biến, kiểm soát tốt bias–variance

Nguyên tắc:
- Tất cả mô hình dùng **cùng feature set**
- Cùng **Stratified K-Fold splits**
- Chỉ đánh giá bằng **cross-validation trên train set**
- Chưa dùng test set


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Stratified K-Fold (dùng chung cho tất cả models)
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

SCORING = "roc_auc"

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Logistic Regression (Scaler + L2)
lr_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]
)

lr_cv_scores = cross_val_score(
    lr_pipeline,
    X_train_fe[FEATURES],
    y_train_prep,
    cv=cv,
    scoring=SCORING
)

print(
    f"Logistic Regression CV AUC: "
    f"{lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}"
)

In [ ]:
! pip install catboost
from catboost import CatBoostClassifier

# CatBoostClassifier
cat_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False
)

cat_cv_scores = cross_val_score(
    cat_model,
    X_train_fe[FEATURES],
    y_train_prep,
    cv=cv,
    scoring=SCORING
)

print(
    f"CatBoost CV AUC: "
    f"{cat_cv_scores.mean():.4f} ± {cat_cv_scores.std():.4f}"
)

In [ ]:
from xgboost import XGBClassifier

# XGBoostClassifier
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=RANDOM_STATE,
    use_label_encoder=False
)

xgb_cv_scores = cross_val_score(
    xgb_model,
    X_train_fe[FEATURES],
    y_train_prep,
    cv=cv,
    scoring=SCORING
)

print(
    f"XGBoost CV AUC: "
    f"{xgb_cv_scores.mean():.4f} ± {xgb_cv_scores.std():.4f}"
)

In [ ]:
cv_summary = pd.DataFrame({
    "Model": ["Logistic Regression", "CatBoost", "XGBoost"],
    "AUC_mean": [
        lr_cv_scores.mean(),
        cat_cv_scores.mean(),
        xgb_cv_scores.mean()
    ],
    "AUC_std": [
        lr_cv_scores.std(),
        cat_cv_scores.std(),
        xgb_cv_scores.std()
    ]
})

cv_summary

**Nhận xét kết quả Cross-Validation**

Kết quả cross-validation cho thấy ba mô hình đều đạt hiệu suất cao và ổn định,
với ROC-AUC dao động quanh mức 0.80.

- **Logistic Regression** đạt AUC trung bình ≈ 0.795 với độ lệch chuẩn rất thấp,
  cho thấy mô hình tuyến tính, khi kết hợp với feature engineering và interaction features,
  vẫn có khả năng nắm bắt tốt xu hướng nguy cơ tim mạch trong dữ liệu.
  Điều này cũng phản ánh tín hiệu tuyến tính mạnh giữa tuổi, huyết áp và BMI.

- **CatBoost** cho kết quả AUC cao nhất (≈ 0.800) và đồng thời có độ dao động nhỏ nhất
  giữa các fold. Điều này cho thấy mô hình boosting có khả năng khai thác tốt
  các mối quan hệ phi tuyến và tương tác phức tạp giữa các biến lâm sàng,
  đồng thời duy trì độ ổn định cao.

- **XGBoost** đạt hiệu suất tương đương CatBoost (AUC ≈ 0.799),
  với độ lệch chuẩn tương tự, cho thấy hai mô hình tree-based đều phù hợp
  với bài toán dữ liệu dạng tabular này.

Nhìn chung, sự khác biệt AUC giữa các mô hình là tương đối nhỏ
và nằm trong phạm vi độ lệch chuẩn của cross-validation.
Do đó, không có mô hình đơn lẻ nào vượt trội một cách áp đảo.
Điều này tạo động lực rõ ràng cho việc thử nghiệm **Soft Voting Ensemble**
ở bước tiếp theo, với kỳ vọng cải thiện hiệu suất một cách ổn định
thông qua việc kết hợp các mô hình có hành vi dự đoán bổ trợ lẫn nhau.


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve, auc

# Hàm vẽ ROC từ OOF predictions
def plot_roc_curves(models, X, y, cv):
    plt.figure(figsize=(8, 6))

    for name, model in models.items():
        # OOF predicted probabilities
        oof_proba = cross_val_predict(
            model,
            X,
            y,
            cv=cv,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]

        fpr, tpr, _ = roc_curve(y, oof_proba)
        roc_auc = auc(fpr, tpr)

        plt.plot(
            fpr,
            tpr,
            label=f"{name} (AUC = {roc_auc:.3f})"
        )

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves (Cross-Validated, Train Set)")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()


models_dict = {
    "Logistic Regression": lr_pipeline,
    "CatBoost": cat_model,
    "XGBoost": xgb_model
}

plot_roc_curves(
    models=models_dict,
    X=X_train_fe[FEATURES],
    y=y_train_prep,
    cv=cv
)

## 7. Voting Classifier (Soft Voting Ensemble)

### 7.1 Động cơ (Rationale)

Ba mô hình cơ sở có đặc tính khác nhau:
- Logistic Regression: học xu hướng tuyến tính, ổn định, dễ calibration
- CatBoost & XGBoost: học quan hệ phi tuyến và interaction phức tạp

Soft voting kết hợp **predicted probabilities** giúp:
- Giảm variance
- Khai thác tính đa dạng (diversity) giữa các mô hình
- Phù hợp với metric ROC-AUC và calibration

### 7.2 Cấu hình được thử nghiệm

- Single models (đã có ở Bước 6)
- Soft voting:
  - LR + CatBoost + XGBoost
  - CatBoost + XGBoost
- (Tuỳ chọn) Weighted soft voting theo CV AUC

Nguyên tắc:
- Tất cả estimators phải có `predict_proba`
- Dùng **cùng CV folds** như các mô hình đơn lẻ
- Chỉ đánh giá trên **train set**


In [ ]:
# Logistic Regression pipeline (giữ nguyên cấu hình)
lr_est = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]
)

# CatBoost
cat_est = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False
)

# XGBoost
xgb_est = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=RANDOM_STATE,
    use_label_encoder=False
)

In [ ]:
# Soft Voting: LR + CatBoost + XGBoost
voting_all = VotingClassifier(
    estimators=[
        ("lr", lr_est),
        ("cat", cat_est),
        ("xgb", xgb_est),
    ],
    voting="soft",
    n_jobs=-1
)

voting_all_cv = cross_val_score(
    voting_all,
    X_train_fe[FEATURES],
    y_train_prep,
    cv=cv,
    scoring=SCORING
)

print(
    f"Soft Voting (LR + Cat + XGB) CV AUC: "
    f"{voting_all_cv.mean():.4f} ± {voting_all_cv.std():.4f}"
)

In [ ]:
# Soft Voting: CatBoost + XGBoost
voting_tree = VotingClassifier(
    estimators=[
        ("cat", cat_est),
        ("xgb", xgb_est),
    ],
    voting="soft",
    n_jobs=-1
)

voting_tree_cv = cross_val_score(
    voting_tree,
    X_train_fe[FEATURES],
    y_train_prep,
    cv=cv,
    scoring=SCORING
)

print(
    f"Soft Voting (Cat + XGB) CV AUC: "
    f"{voting_tree_cv.mean():.4f} ± {voting_tree_cv.std():.4f}"
)

In [ ]:
# Weighted Soft Voting (research bonus)

# Ví dụ: weight theo AUC mean (chuẩn hoá nhẹ)
weights_all = [
    lr_cv_scores.mean(),
    cat_cv_scores.mean(),
    xgb_cv_scores.mean(),
]

voting_all_weighted = VotingClassifier(
    estimators=[
        ("lr", lr_est),
        ("cat", cat_est),
        ("xgb", xgb_est),
    ],
    voting="soft",
    weights=weights_all,
    n_jobs=-1
)

voting_all_weighted_cv = cross_val_score(
    voting_all_weighted,
    X_train_fe[FEATURES],
    y_train_prep,
    cv=cv,
    scoring=SCORING
)

print(
    f"Weighted Soft Voting CV AUC: "
    f"{voting_all_weighted_cv.mean():.4f} ± {voting_all_weighted_cv.std():.4f}"
)

In [ ]:
# Tổng hợp kết quả CV (single + ensemble)
cv_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "CatBoost",
        "XGBoost",
        "Voting (LR+Cat+XGB)",
        "Voting (Cat+XGB)",
        "Weighted Voting (LR+Cat+XGB)"
    ],
    "AUC_mean": [
        lr_cv_scores.mean(),
        cat_cv_scores.mean(),
        xgb_cv_scores.mean(),
        voting_all_cv.mean(),
        voting_tree_cv.mean(),
        voting_all_weighted_cv.mean(),
    ],
    "AUC_std": [
        lr_cv_scores.std(),
        cat_cv_scores.std(),
        xgb_cv_scores.std(),
        voting_all_cv.std(),
        voting_tree_cv.std(),
        voting_all_weighted_cv.std(),
    ]
})

cv_summary

**Nhận xét kết quả Voting Classifier**

Kết quả cross-validation cho thấy cả ba mô hình đơn lẻ đều đạt hiệu suất tốt,
với ROC-AUC trung bình dao động quanh ngưỡng 0.80.
Trong đó, CatBoost là mô hình đơn lẻ có AUC cao nhất
(AUC ≈ 0.8004 ± 0.0025).

Soft Voting Ensemble giữa Logistic Regression, CatBoost và XGBoost
đạt ROC-AUC cao nhất (AUC ≈ 0.8006), tuy nhiên mức cải thiện so với CatBoost
là **rất nhỏ** và nằm trong phạm vi độ lệch chuẩn cross-validation.
Điều này cho thấy ensemble **không mang lại cải thiện đáng kể về mặt thống kê**,
nhưng vẫn cho hiệu suất **ổn định và nhất quán**.

So sánh giữa các cấu hình voting cho thấy:
- Voting (CatBoost + XGBoost) đạt hiệu suất tương đương CatBoost đơn lẻ
- Việc thêm Logistic Regression vào ensemble
  mang lại cải thiện rất nhẹ về AUC trung bình
- Weighted soft voting không cải thiện thêm so với soft voting không trọng số

Nhìn chung, kết quả gợi ý rằng lợi ích chính của voting ensemble trong bài toán này
không nằm ở việc tăng mạnh AUC,
mà ở **việc kết hợp các mô hình có inductive bias khác nhau**
(tuyến tính và phi tuyến),
từ đó giúp mô hình ensemble đạt hiệu suất ổn định
và giảm rủi ro phụ thuộc vào một mô hình đơn lẻ.


## 8. Evaluation

### 8.1 Nguyên tắc đánh giá

- Cross-validation (CV) đã dùng để:
  - So sánh mô hình
  - Ước lượng độ ổn định (mean ± std)
- Test set **chỉ dùng một lần duy nhất** để:
  - Báo cáo performance cuối
  - Đánh giá khả năng tổng quát hóa

### 8.2 Metrics được báo cáo

- **ROC-AUC** (primary)
- **Brier score** (đánh giá calibration)
- **Calibration curve** (so sánh xác suất dự đoán và tần suất thực tế)


In [ ]:
# Chọn model tốt nhất (theo CV)
best_model = voting_all

print("Best model selected: Soft Voting (LR + CatBoost + XGBoost)")

In [ ]:
# Fit trên toàn bộ train set
best_model.fit(
    X_train_fe[FEATURES],
    y_train_prep
)

print("Model fitted on full training data.")

In [ ]:
# Predicted probabilities on test set
y_test_proba = best_model.predict_proba(
    X_test_fe[FEATURES]
)[:, 1]

# ROC-AUC
test_auc = roc_auc_score(y_test_prep, y_test_proba)

# Brier score
test_brier = brier_score_loss(y_test_prep, y_test_proba)

print(f"Test ROC-AUC : {test_auc:.4f}")
print(f"Test Brier   : {test_brier:.4f}")

**Đánh giá hiệu suất trên Test Set (ROC-AUC & Brier Score)**

Mô hình **Soft Voting Ensemble (Logistic Regression + CatBoost + XGBoost)**
đạt **ROC-AUC ≈ 0.797** trên tập test, cho thấy khả năng phân biệt tốt giữa
nhóm có và không có bệnh tim mạch.

So với kết quả cross-validation trên tập train, giá trị AUC chỉ giảm nhẹ
và vẫn nằm trong khoảng dao động (mean ± std), cho thấy mô hình
**tổng quát hóa tốt và không có dấu hiệu overfitting rõ rệt**.

Brier score đạt khoảng **0.183**, phản ánh rằng xác suất dự đoán của mô hình
ở mức **chấp nhận được đối với bài toán y tế**, đồng thời không có dấu hiệu
dự đoán quá tự tin hoặc quá kém ổn định.


In [ ]:
from sklearn.calibration import calibration_curve

# Compute calibration curve
prob_true, prob_pred = calibration_curve(
    y_test_prep,
    y_test_proba,
    n_bins=10,
    strategy="quantile"
)

# Plot
plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker="o", label="Model")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed fraction of positives")
plt.title("Calibration Curve (Test Set)")
plt.legend()
plt.grid(True)
plt.show()

**Đánh giá Calibration trên Test Set**

Calibration curve trên tập test cho thấy đường dự đoán của mô hình
nằm tương đối gần đường calibration lý tưởng (đường chéo).

Cụ thể:
- Ở vùng xác suất thấp đến trung bình (khoảng 0.2–0.7),
  mô hình dự đoán khá sát với tần suất mắc bệnh thực tế
- Ở vùng xác suất cao, mô hình có xu hướng **dự đoán hơi thận trọng**
  (under-confident nhẹ), điều này thường được xem là an toàn hơn
  trong bối cảnh đánh giá rủi ro y tế

Nhìn chung, mô hình thể hiện **calibration hợp lý**, phù hợp để sử dụng
trong các bước phân tích ngưỡng quyết định và phân tầng rủi ro
ở các phần tiếp theo.


## 9. Decision Threshold Analysis

### 9.1 Vì sao cần phân tích threshold?

ROC-AUC đánh giá khả năng phân biệt tổng thể, nhưng:
- Trong thực tế, quyết định lâm sàng cần một **ngưỡng xác suất cụ thể**
- Threshold ảnh hưởng trực tiếp đến:
  - Sensitivity / Specificity
  - Precision / Recall

### 9.2 Nguyên tắc chống data leakage

- Threshold **KHÔNG được chọn trên test set**
- Threshold được xác định bằng:
  - Cross-validation predictions (out-of-fold)
  - Hoặc validation set tách từ train

Trong bài này, threshold được chọn bằng **OOF predicted probabilities**
từ cross-validation trên train set.


In [ ]:
from sklearn.model_selection import cross_val_predict

# OOF predicted probabilities (train-only)
y_train_oof_proba = cross_val_predict(
    best_model,
    X_train_fe[FEATURES],
    y_train_prep,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("OOF predictions shape:", y_train_oof_proba.shape)

In [ ]:
# Chọn threshold theo Youden’s J statistic

'Youden’s J = Sensitivity + Specificity − 1'

from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_train_prep, y_train_oof_proba)

youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
best_threshold = thresholds[best_idx]

print(f"Selected threshold (Youden J, OOF): {best_threshold:.3f}")


In [ ]:
# Đánh giá confusion matrix trên TRAIN (OOF)

from sklearn.metrics import confusion_matrix, classification_report

# OOF predicted class
y_train_oof_pred = (y_train_oof_proba >= best_threshold).astype(int)

cm = confusion_matrix(y_train_prep, y_train_oof_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

cm_df

**Nhận xét Confusion Matrix (OOF – Train set)**


Ma trận nhầm lẫn được xây dựng trên **out-of-fold (OOF) predictions của tập huấn luyện**,
sử dụng ngưỡng xác suất được chọn bằng **Youden’s J statistic**.
Do đó, kết quả này phản ánh hành vi mô hình trên dữ liệu *chưa từng thấy trong từng fold*,
giúp tránh hiện tượng overfitting.

Các quan sát chính:
- Mô hình dự đoán đúng phần lớn cả hai lớp:
  - True Negative (Actual 0 → Predicted 0): 21,184
  - True Positive (Actual 1 → Predicted 1): 19,123
- Số False Negative (bỏ sót bệnh nhân có bệnh): 8,013
- Số False Positive (dự đoán nhầm người khỏe thành bệnh): 6,476

Điều này cho thấy tại ngưỡng hiện tại:
- Mô hình ưu tiên **cân bằng giữa sensitivity và specificity**
- Không quá thiên lệch về một lớp, phù hợp với mục tiêu sàng lọc nguy cơ tim mạch
  (không bỏ sót quá nhiều ca bệnh, nhưng cũng không tạo quá nhiều cảnh báo giả)

Lưu ý quan trọng:
- Confusion matrix này **không sử dụng test set**
- Ngưỡng quyết định được chọn hoàn toàn trên train set thông qua OOF predictions,
đảm bảo **không xảy ra data leakage**


In [ ]:
print(
    classification_report(
        y_train_prep,
        y_train_oof_pred,
        digits=3
    )
)

**Nhận xét Classification Report (OOF – Train set)**

Báo cáo classification được tính toán trên **OOF predicted classes** của tập huấn luyện,
tại ngưỡng xác suất đã được xác định ở bước Decision Threshold Analysis.

Đối với lớp không mắc bệnh (class 0):
- Precision: 0.726
- Recall: 0.766  
→ Mô hình nhận diện khá tốt các trường hợp không mắc bệnh,
với tỷ lệ dự đoán đúng cao và ít báo động giả.

Đối với lớp mắc bệnh tim mạch (class 1):
- Precision: 0.747
- Recall: 0.705  
→ Mô hình phát hiện được khoảng 70.5% các trường hợp có bệnh,
đồng thời giữ mức precision tương đối cao.

Các chỉ số tổng quát:
- Accuracy: 0.736
- Macro average F1-score: 0.735
- Weighted average F1-score: 0.735

Nhận xét chung:
- Precision và recall giữa hai lớp khá cân bằng
- Không có dấu hiệu mô hình bị lệch mạnh về một phía
- Kết quả phù hợp với mục tiêu sàng lọc nguy cơ,
nơi việc cân bằng giữa phát hiện ca bệnh và hạn chế cảnh báo giả là quan trọng

Quan trọng nhất:
- Các chỉ số này **chỉ mang tính tham khảo hành vi tại ngưỡng**
- Hiệu suất chính thức của mô hình vẫn được đánh giá bằng ROC-AUC và calibration
- Test set **chưa và sẽ không được sử dụng** để chọn ngưỡng


## 10. Post-hoc Risk Stratification

### 10.1 Mục tiêu

Mục tiêu của bước này là:
- Diễn giải mô hình dưới góc nhìn lâm sàng
- Xác định các **ngưỡng rủi ro (risk thresholds)** theo:
  - Tuổi
  - Huyết áp
  - BMI

Quan trọng:
- Đây là **phân tích hậu kiểm (post-hoc)**
- KHÔNG dùng để train model
- KHÔNG dùng test set để tìm ngưỡng

### 10.2 Nguyên tắc chống data leakage

- Toàn bộ phân tích dựa trên:
  - **Out-of-fold (OOF) predicted probabilities**
  - Tập train
- Test set **tuyệt đối không tham gia** vào việc:
  - Chia bin
  - Tìm inflection point


In [ ]:
# DataFrame phục vụ post-hoc analysis (train-only)
posthoc_df = X_train_fe[[
    "age_years",
    "ap_hi",
    "bmi"
]].copy()

posthoc_df["y_true"] = y_train_prep.values
posthoc_df["risk_proba"] = y_train_oof_proba

posthoc_df.head()

In [ ]:
# Age bins (fixed, dễ giải thích)
age_bins = [30, 40, 50, 60, 70, 80]
posthoc_df["age_bin"] = pd.cut(
    posthoc_df["age_years"],
    bins=age_bins,
    right=False
)

age_risk = (
    posthoc_df
    .groupby("age_bin")["risk_proba"]
    .mean()
    .reset_index()
)

age_risk

In [ ]:
# BMI classes
def bmi_class_verbose(bmi):
    if bmi < 18.5:
        return "Underweight (<18.5)"
    elif bmi < 25:
        return "Normal (18.5–24.9)"
    elif bmi < 30:
        return "Overweight (25–29.9)"
    else:
        return "Obese (≥30)"


posthoc_df["bmi_class"] = posthoc_df["bmi"].apply(bmi_class_verbose)

bmi_risk = (
    posthoc_df
    .groupby("bmi_class")["risk_proba"]
    .mean()
    .reset_index()
)

bmi_risk

In [ ]:
# Blood pressure stages
def bp_stage_verbose(ap_hi):
    if ap_hi < 120:
        return "Normal (<120 mmHg)"
    elif ap_hi < 130:
        return "Elevated (120–129 mmHg)"
    elif ap_hi < 140:
        return "Stage 1 HTN (130–139 mmHg)"
    else:
        return "Stage 2 HTN (≥140 mmHg)"


posthoc_df["bp_stage"] = posthoc_df["ap_hi"].apply(bp_stage_verbose)

bp_risk = (
    posthoc_df
    .groupby("bp_stage")["risk_proba"]
    .mean()
    .reset_index()
)

bp_risk

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.barplot(
    data=age_risk,
    x="age_bin",
    y="risk_proba",
    ax=axes[0]
)
axes[0].set_title("Mean Predicted Risk by Age Bin")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(
    data=bmi_risk,
    x="bmi_class",
    y="risk_proba",
    ax=axes[1]
)
axes[1].set_title("Mean Predicted Risk by BMI Class")

sns.barplot(
    data=bp_risk,
    x="bp_stage",
    y="risk_proba",
    ax=axes[2]
)
axes[2].set_title("Mean Predicted Risk by BP Stage")

plt.tight_layout()
plt.show()

## 11. Explainability

### 11.1 Mục tiêu

Mục tiêu của phần này là:
- Hiểu **vì sao mô hình dự đoán như vậy**
- Kết nối kết quả mô hình với **ý nghĩa lâm sàng**

Chiến lược:
- Logistic Regression → Odds Ratio (giải thích tuyến tính)
- CatBoost / XGBoost → SHAP values (giải thích phi tuyến)

Lưu ý:
- Đây là phân tích hậu kiểm (post-hoc)
- Không dùng để chọn feature hay tune model


In [ ]:
# Fit Logistic Regression trên toàn bộ train
lr_pipeline.fit(
    X_train_fe[FEATURES],
    y_train_prep
)

# Lấy hệ số
coef = lr_pipeline.named_steps["lr"].coef_[0]
features = FEATURES

or_df = pd.DataFrame({
    "feature": features,
    "coef": coef,
    "odds_ratio": np.exp(coef)
}).sort_values("odds_ratio", ascending=False)

or_df

**Nhận xét các yếu tố làm TĂNG nguy cơ tim mạch (Logistic Regression)**

Bảng trên trình bày các biến có **odds ratio > 1**, tức là các yếu tố
**làm tăng xác suất mắc bệnh tim mạch** khi giá trị của chúng tăng.

Các yếu tố nổi bật gồm:

- **Huyết áp tâm thu (ap_hi)** có ảnh hưởng mạnh nhất  
  (odds ratio ≈ 1.59), nghĩa là khi huyết áp tâm thu tăng,
  nguy cơ mắc bệnh tim tăng rõ rệt.
  Đây là kết quả hoàn toàn phù hợp với kiến thức y học.

- **Huyết áp tâm trương (ap_lo)** và **pulse pressure**
  cũng có odds ratio cao (> 1.4), cho thấy huyết áp
  không chỉ quan trọng ở một chỉ số đơn lẻ,
  mà cả sự chênh lệch giữa tâm thu và tâm trương
  cũng mang ý nghĩa rủi ro tim mạch.

- **Cholesterol** và **tuổi (age_years)** đều là các yếu tố nguy cơ quen thuộc,
  với odds ratio lần lượt khoảng 1.39 và 1.38.
  Điều này phản ánh rằng mô hình đã học được
  các mối quan hệ lâm sàng hợp lý.

- **BMI** có ảnh hưởng dương nhưng yếu hơn (OR ≈ 1.16),
  cho thấy béo phì là yếu tố nguy cơ,
  nhưng tác động của BMI đơn lẻ không mạnh bằng huyết áp hoặc tuổi.

Nhìn chung, các yếu tố làm tăng nguy cơ đều là
những biến lâm sàng có ý nghĩa rõ ràng,
giúp Logistic Regression dễ diễn giải và đáng tin cậy
trong bối cảnh y tế.


In [ ]:
# Top positive / negative effects
top_positive = or_df.head(8)
top_negative = or_df.tail(8)

top_positive, top_negative

**Nhận xét các yếu tố làm GIẢM hoặc ĐIỀU CHỈNH nguy cơ tim mạch**

Bảng này trình bày các biến có **odds ratio < 1**,
tức là các yếu tố có xu hướng **làm giảm nguy cơ**
hoặc đóng vai trò **điều chỉnh mối quan hệ giữa các biến chính**.

Một số điểm đáng chú ý:

- Các **interaction features** như:
  - age_ap_hi  
  - bmi_ap_hi  
  - age_bmi  

  đều có odds ratio < 1.
  Điều này cho thấy khi đã xét đến sự kết hợp giữa tuổi, BMI và huyết áp,
  tác động của từng biến riêng lẻ có thể bị điều chỉnh.
  Đây là hiện tượng thường gặp khi mô hình đã học được
  các mối quan hệ phức tạp giữa các yếu tố nguy cơ.

- Các biến hành vi như **smoke**, **alco** và **active**
  có odds ratio nhỏ hơn 1 nhưng giá trị gần 1,
  cho thấy tác động của chúng trong mô hình này là tương đối yếu
  so với các yếu tố sinh lý học như tuổi và huyết áp.

- Việc một số biến hành vi có odds ratio < 1
  không nhất thiết mang ý nghĩa bảo vệ tuyệt đối,
  mà phản ánh rằng trong dữ liệu này,
  ảnh hưởng của chúng bị lấn át bởi các yếu tố nguy cơ chính
  hoặc có thể chịu nhiễu do dữ liệu tự báo cáo.

Tổng thể, bảng này cho thấy Logistic Regression
không chỉ học các yếu tố nguy cơ trực tiếp,
mà còn học được cách các yếu tố đó tương tác và điều chỉnh lẫn nhau,
giúp mô hình ổn định hơn khi có interaction features.


In [ ]:
# Tree-based models – SHAP
"""
SHAP giúp:

- Giải thích prediction của CatBoost / XGBoost

- Phát hiện interaction phi tuyến mà LR không nắm được
"""
import shap

# Fit CatBoost trên full train (phục vụ explainability)
cat_est.fit(
    X_train_fe[FEATURES],
    y_train_prep
)

# SHAP explainer cho CatBoost
import shap
explainer = shap.TreeExplainer(cat_est)

# Tính SHAP values trên train set
shap_values = explainer.shap_values(
    X_train_fe[FEATURES]
)

print("SHAP values computed for CatBoost.")

In [ ]:
# SHAP summary plot
shap.summary_plot(
    shap_values,
    X_train_fe[FEATURES],
    plot_type="dot",
    show=True
)

In [ ]:
# SHAP dependence (interaction insight)

# Các biến quan trọng để phân tích
for feature in ["age_years", "ap_hi", "bmi", "pulse_pressure"]:
    shap.dependence_plot(
        feature,
        shap_values,
        X_train_fe[FEATURES],
        show=True
    )


SHAP analysis cho thấy tuổi, huyết áp tâm thu, BMI và pulse pressure
là các feature có ảnh hưởng lớn nhất đến dự đoán của mô hình.

Các SHAP dependence plots cho thấy mối quan hệ phi tuyến,
đặc biệt là khi kết hợp tuổi cao với huyết áp hoặc BMI cao,
phù hợp với hiểu biết y học hiện tại.

In [ ]:
# Explainability cho Voting (mức khái niệm)
"VotingClassifier không có SHAP trực tiếp, nên giải thích ở mức phân phối."

# Distribution of predicted probabilities
plt.figure(figsize=(6, 4))
sns.kdeplot(y_train_oof_proba, fill=True)
plt.title("Distribution of OOF Predicted Risk (Voting Model)")
plt.xlabel("Predicted probability")
plt.ylabel("Density")
plt.show()

**Nhận xét phân phối xác suất dự đoán (OOF – Voting Model):**

Phân phối xác suất dự đoán từ out-of-fold (OOF) predictions cho thấy mô hình voting
tạo ra một phân bố **không đơn đỉnh (multi-modal)**, phản ánh khả năng tách nhóm
đối tượng có nguy cơ thấp và nguy cơ cao.

Cụ thể:
- Một cụm xác suất tập trung ở vùng thấp (≈ 0.1–0.4), tương ứng với nhóm bệnh nhân
  có nguy cơ tim mạch thấp.
- Một cụm khác rõ rệt ở vùng xác suất cao (≈ 0.7–0.9), đại diện cho nhóm nguy cơ cao.
- Vùng trung gian (≈ 0.45–0.6) đóng vai trò như vùng “không chắc chắn” (uncertainty region),
  nơi mô hình khó phân biệt rõ ràng giữa hai lớp.

Hình dạng phân phối này cho thấy:
- Mô hình ensemble có khả năng **phân biệt rủi ro tốt**, không bị dồn xác suất quanh 0.5.
- Soft voting giúp tạo ra xác suất dự đoán mượt và ổn định hơn, hỗ trợ tốt cho
  việc lựa chọn ngưỡng quyết định (decision threshold) và phân tầng rủi ro hậu kiểm.

Phân tích này được thực hiện hoàn toàn trên **OOF predictions của tập huấn luyện**,
do đó không gây rò rỉ dữ liệu và chỉ mang tính diễn giải (post-hoc analysis).



## 12. Discussion & Conclusion (Rewritten – Data-driven)

### **12.1 So sánh hiệu suất và tính ổn định của các mô hình**

Kết quả cross-validation cho thấy cả ba mô hình đơn lẻ đều đạt hiệu suất cao và ổn định, với ROC-AUC trung bình xấp xỉ 0.80.
Cụ thể, Logistic Regression đạt AUC = **0.7953 ± 0.0029**, cho thấy khi kết hợp continuous modeling và interaction features, mô hình tuyến tính vẫn nắm bắt tốt xu hướng nguy cơ tim mạch trong dữ liệu. Điều này phản ánh sự tồn tại của tín hiệu tuyến tính mạnh giữa tuổi, huyết áp và BMI.

CatBoost là mô hình đơn lẻ có hiệu suất cao nhất với **AUC = 0.8004 ± 0.0025**, đồng thời có độ lệch chuẩn thấp nhất giữa các fold. Điều này cho thấy CatBoost khai thác hiệu quả các mối quan hệ phi tuyến và tương tác phức tạp giữa các biến lâm sàng, đồng thời duy trì độ ổn định cao.

XGBoost đạt hiệu suất tương đương CatBoost (**AUC = 0.7994 ± 0.0028**), xác nhận rằng các mô hình tree-based nói chung là phù hợp với dữ liệu dạng tabular trong bài toán dự đoán bệnh tim mạch.

Nhìn chung, sự khác biệt AUC giữa các mô hình đơn lẻ là nhỏ và nằm trong phạm vi độ lệch chuẩn của cross-validation. Do đó, không có mô hình nào vượt trội một cách áp đảo, tạo tiền đề hợp lý cho việc thử nghiệm ensemble learning.


### **12.2 Hiệu quả thực tế của Soft Voting Ensemble**

Soft Voting Ensemble giữa Logistic Regression, CatBoost và XGBoost đạt **ROC-AUC = 0.8006 ± 0.0029**, cao hơn nhẹ so với mô hình đơn lẻ tốt nhất (CatBoost). Tuy nhiên, mức cải thiện này là rất nhỏ (≈ +0.0002 AUC) và hoàn toàn nằm trong độ lệch chuẩn cross-validation.

Kết quả này cho thấy **ensemble không mang lại cải thiện có ý nghĩa thống kê về mặt ROC-AUC**, nhưng vẫn đạt được hiệu suất **ổn định và nhất quán**. So sánh các cấu hình voting cho thấy:

* Voting (CatBoost + XGBoost) cho kết quả gần tương đương CatBoost đơn lẻ
* Việc thêm Logistic Regression vào ensemble giúp cải thiện rất nhẹ AUC trung bình
* Weighted soft voting không mang lại lợi ích bổ sung so với soft voting không trọng số

Do đó, lợi ích chính của voting ensemble trong nghiên cứu này **không nằm ở việc tối đa hóa AUC**, mà ở việc **giảm rủi ro phụ thuộc vào một mô hình đơn lẻ**, thông qua việc kết hợp các mô hình có inductive bias khác nhau (tuyến tính và phi tuyến).

### **12.3 Khả năng tổng quát hóa và calibration**

Trên tập test, mô hình Soft Voting Ensemble đạt **ROC-AUC = 0.7970**, chỉ giảm nhẹ so với kết quả cross-validation trên tập train và vẫn nằm trong khoảng dao động (mean ± std). Điều này cho thấy mô hình có khả năng tổng quát hóa tốt và không có dấu hiệu overfitting rõ rệt.

Brier score trên test đạt **0.1825**, phản ánh rằng xác suất dự đoán của mô hình ở mức chấp nhận được đối với bài toán y tế. Calibration curve cho thấy mô hình dự đoán khá sát với đường calibration lý tưởng ở vùng xác suất thấp đến trung bình (≈ 0.2–0.7), và có xu hướng **under-confident nhẹ ở vùng xác suất cao**. Trong bối cảnh sàng lọc nguy cơ tim mạch, hành vi này thường được xem là an toàn hơn so với over-confidence.


### **12.4 Ngưỡng quyết định và hành vi phân loại**

Ngưỡng xác suất được lựa chọn bằng **Youden’s J statistic** trên out-of-fold predictions của tập huấn luyện, với giá trị **threshold ≈ 0.492**. Việc lựa chọn ngưỡng hoàn toàn dựa trên train set giúp đảm bảo không xảy ra data leakage.

Tại ngưỡng này, mô hình đạt:

* Recall cho lớp mắc bệnh ≈ **0.705**
* Precision cho lớp mắc bệnh ≈ **0.747**

Các chỉ số này cho thấy mô hình đạt sự cân bằng hợp lý giữa sensitivity và specificity, phù hợp với mục tiêu sàng lọc nguy cơ tim mạch, nơi việc bỏ sót quá nhiều ca bệnh hoặc tạo quá nhiều cảnh báo giả đều không mong muốn.



### **12.5 Insight lâm sàng từ post-hoc risk stratification**

Phân tích hậu kiểm dựa trên out-of-fold predicted probabilities cho thấy các xu hướng rủi ro rõ ràng và nhất quán với kiến thức y học hiện tại:

* Nguy cơ tim mạch tăng mạnh sau mốc tuổi **50–60**, với mean predicted risk tăng từ ≈ **0.37** (40–50) lên ≈ **0.52** (50–60) và ≈ **0.66** (60–70)
* Nhóm **BMI ≥ 30 (obese)** có mean predicted risk cao nhất (**≈ 0.63**), cao hơn rõ rệt so với nhóm BMI bình thường (**≈ 0.40**)
* Nhóm **Stage 2 hypertension (SBP ≥ 140 mmHg)** có predicted risk rất cao (**≈ 0.83**), vượt trội so với các nhóm huyết áp thấp hơn

Các ngưỡng này mang tính diễn giải và không được sử dụng để huấn luyện mô hình, nhưng cung cấp insight có giá trị về cách mô hình học được các yếu tố nguy cơ chính.


### **12.6 Hạn chế và hướng phát triển**

Nghiên cứu vẫn tồn tại một số hạn chế:

* Dữ liệu chỉ bao gồm các đặc trưng lâm sàng cơ bản, chưa có xét nghiệm chuyên sâu
* Dữ liệu tự báo cáo có thể chứa nhiễu
* Chưa có external validation trên tập dữ liệu độc lập
* Calibration có thể được cải thiện thêm bằng các kỹ thuật hậu xử lý

Trong tương lai, có thể mở rộng nghiên cứu bằng cách thử nghiệm stacking hoặc blending, áp dụng CalibratedClassifierCV cho mô hình tốt nhất, hoặc đánh giá mô hình trên các bộ dữ liệu ngoài.



### **12.7 Kết luận**

Nghiên cứu cho thấy việc kết hợp Logistic Regression, CatBoost và XGBoost bằng Soft Voting Ensemble **không tạo ra cải thiện lớn về ROC-AUC**, nhưng mang lại hiệu suất ổn định và khả năng tổng quát hóa tốt. Quan trọng hơn, mô hình không chỉ đạt hiệu quả phân loại hợp lý mà còn cung cấp các insight diễn giải được về vai trò của tuổi, huyết áp và BMI trong nguy cơ tim mạch.

Cách tiếp cận này nhấn mạnh rằng trong các bài toán y tế, **sự ổn định, khả năng diễn giải và calibration** có thể quan trọng không kém việc tối đa hóa một chỉ số hiệu suất đơn lẻ.

